In [ ]:
%pip install -qU google-generativeai langgraph tavily
%pip install -qU tavily-python

In [ ]:
%pip install -qU ddgs

In [ ]:
import os
import google.generativeai as genai
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [ ]:
client = TavilyClient(api_key=TAVILY_API_KEY)
result = client.search("Que son multiagentes de Inteligencia Artificial?", 
                       include_answer=True)

result["answer"]

In [ ]:
ciudad = "Tulum"

query = f"""
Enumere los 5 principales restaurantes en {ciudad}, según evaluaciones recientes en TripAdvisor o sitios similares de turismo.
Para cada restaurante, indique:
- Tipo de cocina (ej.: regional, italiana, japonesa)
- Una breve descripción (máx. 2 líneas)
- Calificación promedio (si está disponible)
- Rango de precios

Responda únicamente con datos actualizados y relevantes para turistas.
"""

In [ ]:
from ddgs import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        raise e

for link in search(query):
    print(link)

In [ ]:
%pip install -qU bs4

In [ ]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

def scrape_restaurantes_info(url):
    if not url:
        print("Error: URL vacia o no localizada.")
        return None

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, como Gecko) Chrome/138.0.0.0 Safari/537.36',
        'Accept-Language': 'es-MX,es;q=0.9,en;q=0.8'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error al cargar la página {url}: {e}")
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

search_results = search(query)

if search_results:
    url = search_results[0]
else:
    url = None

soup = scrape_restaurantes_info(url)

print(f"Website: {url}\n\n")

if soup:
    print(str(soup.body.prettify())[:50000])
else:
    print("No fue posible raspar el contenido de la página o la URL no fue encontrada.")

### Explorando alternativas de búsqueda con Tavily

In [ ]:
import os
import re
from tavily import TavilyClient

api_key = os.environ.get("TAVILY_API_KEY")
if not api_key:
    raise ValueError("la clave TAVILY_API_KEY no fue encontrada.")

cliente_tavily = TavilyClient(api_key=api_key)

ciudad = "Tulum"
tavily_query = f"restaurantes en {ciudad} tripadvisor con mayor cantidad de reviews y rango de precios"

print("Iniciando la búsqueda agéntica por URLs de Tripadvisor con Tavily...")
tripadvisor_url = None

try:
    tavily_results = cliente_tavily.search(query=tavily_query, max_results=5)

    if tavily_results and tavily_results["results"]:
        print(f"Tavily encontró {len(tavily_results['results'])} resultados. Analizando...")
        for result in tavily_results["results"]:
            url = result["url"]
            if "tripadvisor.com" in url or "tripadvisor.com.mx" in url:
                tripadvisor_url = url
                break
    if not tripadvisor_url:
        print("Ninguna URL relevante de Tripadvisor fue encontrada en los primeros resultados.")
    else:
        print("Tavily no encontró resultados para la búsqueda agéntica.")
except Exception as e:
    print(f"Error en la búsqueda agéntica con Tavily: {e}. Verifica tu clave de API o tu conexión.")

if tripadvisor_url:
    clean_url = re.sub(r'-oa\d+-', '-', tripadvisor_url)
    tripadvisor_url = clean_url
    print(f"✅ URL encontrada limpia de paginación.")

print("*" * 50)
print(f"URL Final de Tripadvisor para el raspado: {tripadvisor_url if tripadvisor_url else 'NO ENCONTRADO'}")
print("*" * 50)

## 02 Búsqueda Agéntica y Webscraping

%pip install -qU selenium
%pip install -qU webdriver-manager

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.common.exceptions import WebDriverException, TimeoutException

def scrape_restaurantes_info(url):
    if not url:
        print("Error: URL vacia o no localizada para el raspado.")
        return None
    
    try:
        service = Service(ChromeDriverManager().install())
        options = webdriver.ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
        
        driver = webdriver.Chrome(service=service, options=options)
        driver.set_page_load_timeout(30)
        
    except Exception as e:
        print(f"Error al inicializar el driver de Selenium: {e}")
        return None
    try:
        print(f"Tratando cargar la página con Selenium: {url}")
        driver.get(url)
        
        driver.implicitly_wait(10)
        
        response_text = driver.page_source
    except TimeoutException:
        print(f"Error de limite de tiempo al cargar la página ({url}).")
        return None
    except WebDriverException as e:
        print(f"Error al cargar la página {url} con Selenium: {e}. Puede ser debido a un bloqueo o a que la URL no es válida.")
        return None
    finally:
        driver.quit()
        
    soup = BeautifulSoup(response_text, 'html.parser')
    return soup
    soup_tripadvisor = None

    if 'tripadvisor_url' in locals() and tripadvisor_url:
        print(f"\nTratando raspar la página identificada: {tripadvisor_url}")
        soup_tripadvisor = scrape_restaurantes_info(tripadvisor_url)
    
    if soup_tripadvisor:
        print("HTML de la página de Tripadvisor obtenido con éxito!")
        page_title_tag = soup_tripadvisor.find('title')
        if page_title_tag:
            print(f"Título de la página: {page_title_tag.get_text(strip=True)}")
        else:
            print("No fue posible encontrar el título de la página.")
    else:
        print("Falla al raspar la página de Tripadvisor. Verifica la URL o si la página está bloqueada.")
    else:
        print("No existe URL de Tripadvisor válida para el raspado (obtenido en el bloque 1).")

    print("*" * 50)

### Inspeccionando y extrayendo información de restaurantes 

In [ ]:
from bs4 import BeautifulSoup
import re # Importar regex para una extracción más limpia

# Este es el HTML completo que me proporcionaste - Como si estuviéramos enseñando al agente de búsqueda
html_completo_ejemplos = """
<!DOCTYPE html><html lang="en-US"><head><link rel="icon" id="favicon" type="image/x-icon" href="https://static.tacdn.com/favicon.ico?v2"><meta name="description" content="Tulum Mid Range Restaurants: See 38,727 Tripadvisor traveler reviews of Mid Range Restaurants in Tulum."><meta name="viewport" content="width=device-width, initial-scale=1.0"><link rel="canonical" href="https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html"><meta property="og:url" content="https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html">
"""

# Creando un objeto BeautifulSoup a partir del HTML completo
soup_completo = BeautifulSoup(html_completo_ejemplos, 'html.parser')

# Encontrar todos los bloques de restaurante
# La clase 'tbrcR' combinada con otras parece ser un buen selector para cada "tarjeta" de restaurante
restaurant_blocks = soup_completo.find_all('div', {'data-automation': 'restaurantCard'})

print("Iniciando la extracción detallada de cada restaurante:")
restaurantes_detallados = [] # Lista para almacenar los datos estructurados

for block in restaurant_blocks:
    # --- Selectores para la información DENTRO de cada bloque de restaurante ---

    # 1. Nombre del Restaurante
    # Pistas: El nombre está dentro de una <a> que tiene las clases 'BMQDV' y 'ukgos'.
    # El texto real del nombre está dentro de una <div> que es hija de esa <a>.
    # Ejemplo HTML: <a class="BMQDV ... ukgos"><div class="biGQs ...">1. Nombre del Restaurante</div></a>
    nombre_link_tag = block.find('a', class_=lambda c: c and 'BMQDV' in c and 'ukgos' in c)
    nombre = "Nombre no encontrado"
    if nombre_link_tag:
        # La div específica que contiene el texto del nombre
        nombre_div_tag = nombre_link_tag.find('div', class_=lambda c: c and 'biGQs' in c and 'OgHOe' in c)
        if nombre_div_tag:
            nombre = nombre_div_tag.get_text(strip=True)

    # 2. Calificacion (ej., 4.9)
    # Pistas: Encontrar la div con data-automation="bubbleRatingValue" y tomar el texto del span interno
    calificacion_tag = block.find('div', {'data-automation': 'bubbleRatingValue'})
    calificacion = calificacion_tag.find('span').get_text(strip=True) if calificacion_tag else "Calificación no encontrada"

    # 3. Cantidad de Reseñas
    # Pistas: Está en una <div> con el atributo 'data-automation'="bubbleReviewCount".
    # El texto real de la cantidad está dentro de un <span> que es hijo de esa <div>.
    # Usamos regex para extraer solo los números.
    review_count_tag = block.find('div', {'data-automation': 'bubbleReviewCount'})
    cantidad_resenas = "Cantidad de reseñas no encontrada"
    if review_count_tag:
        text_content = review_count_tag.get_text(strip=True) # (2.529 reseñas) -> "2.529"
        # Extraer solo los números de la cadena
        match = re.search(r'[\d\.]+', text_content)
        if match:
            cantidad_resenas = match.group(0).replace('.', '') # Elimina el punto para convertir a entero

    # 4. Tipo de Cocina (ej: Mexicana, Bar)
    # Pistas: Es un <span> dentro de un <div> con clases 'Zvrsw N G biqBm'.
    # El texto puede estar directamente en el <span> o en un <span> anidado.
    # Intentaremos tomar el primer span que no contenga un SVG o un enlace.
    cuisine_tag = block.find('div', class_=lambda c: c and 'Zvrsw' in c and 'biqBm' in c)
    tipo_cocina = []
    if cuisine_tag:
        # Encontrar todos los spans que son hijos directos o casi directos y que contienen texto relevante
        spans = cuisine_tag.find_all('span', class_=lambda c: c and 'biGQs' in c and 'ZNjNf' in c)
        for span in spans:
            # Excluir spans que contienen SVGs (íconos) o que parecen ser contadores de reseñas
            # y tomar solo el texto que no sea de cantidad de reseñas ni "Cerrado ahora" ni "Menú"
            if not span.find('svg') and 'reseñas' not in span.get_text() and 'Cerrado ahora' not in span.get_text():
                cuisine_text = span.get_text(strip=True)
                if cuisine_text:
                    tipo_cocina.append(cuisine_text)
        if not tipo_cocina:
            tipo_cocina.append("Tipo de cocina no encontrado")
    tipo_cocina = ", ".join(tipo_cocina) if tipo_cocina else "Tipo de cocina no encontrado"
    tipo_cocina = tipo_cocina[0] if isinstance(tipo_cocina, list) and len(tipo_cocina) > 0 else tipo_cocina
    tipo_cocina = tipo_cocina.split(',')[0] if tipo_cocina else tipo_cocina
    tipo_cocina = tipo_cocina[0]

    # 5. Rango de Precios (ej: $$ - $$$)
    # Pistas: Está en un <span> con las mismas clases del tipo de cocina, pero es el que no tiene texto
    # y que corresponde al patrón de precio.
    rango_precio = "Rango de precios no encontrado"
    if cuisine_tag:
        for span in cuisine_tag.find_all('span', class_=lambda c: c and 'biGQs' in c and 'ZNjNf' in c):
            text_content = span.get_text(strip=True)
            if re.match(r'^\$+', text_content): # Verifica si el texto es uno o más símbolos de dólar
                rango_precio = text_content
                break

    # 6. Estado de Funcionamiento (Abierto/Cerrado)
    # Pistas: Está dentro de una div con clase 'Pwlnc f lFNDg', y el texto real en el span interno.
    status_funcionamiento_tag = block.find('div', class_=lambda c: c and 'Pwlnc' in c and 'lFNDg' in c)
    status_funcionamiento = status_funcionamiento_tag.find('span').get_text(strip=True) if status_funcionamiento_tag else "Estado de funcionamiento no encontrado"

    # 7. URL del Restaurante
    # Pistas: la etiqueta <a> principal que envuelve todo el bloque del restaurante.
    # El atributo 'href' contiene la ruta relativa. Necesitamos concatenarla con la base de Tripadvisor.
    url_tag = block.find('a', class_=lambda c: c and 'BMQDV' in c and 'ParlG' in c)
    url_base = "https://www.tripadvisor.com.mx"
    url_restaurante = url_base + url_tag['href'] if url_tag and 'href' in url_tag.attrs else "URL no encontrada"

    # 8. URL de la Imagen Principal
    # Pistas: La primera etiqueta <picture> dentro del carrusel, y la etiqueta <img> dentro de ella.
    # El atributo 'src' contiene la URL de la imagen.
    image_tag = block.find('div', class_='IdURt').find('img') if block.find('div', class_='IdURt') else None
    url_imagen = image_tag['src'] if image_tag and 'src' in image_tag.attrs else "URL de la imagen no encontrada"
    
    # Almacenar los datos en un diccionario
    restaurante_info = {
        "Nombre": nombre,
        "Calificacion": calificacion,
        "Cantidad_Resenas": cantidad_resenas,
        "Tipo_Cocina": tipo_cocina,
        "Rango_Precio": rango_precio,
        "Estado_Funcionamiento": status_funcionamiento,
        "URL_Restaurante": url_restaurante,
        "URL_Imagen_Principal": url_imagen
    }
    restaurantes_detallados.append(restaurante_info)

# Imprimir los resultados para verificación
for i, restaurante in enumerate(restaurantes_detallados):
    print(f"\n--- Restaurante #{i+1} ---")
    for key, value in restaurante.items():
        print(f"{key}: {value}")

### Procesando y verificando la extracción de datos

In [ ]:
from bs4 import BeautifulSoup

html_do_tripadvisor = """
<!DOCTYPE html><html lang="en-US"><head><link rel="icon" id="favicon" type="image/x-icon" href="https://static.tacdn.com/favicon.ico?v2"><meta name="description" content="Tulum Mid Range Restaurants: See 38,727 Tripadvisor traveler reviews of Mid Range Restaurants in Tulum."><meta name="viewport" content="width=device-width, initial-scale=1.0"><link rel="canonical" href="https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html"><meta property="og:url" content="https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html">
"""

soup_tripadvisor = BeautifulSoup(html_do_tripadvisor, 'html.parser')

print("\nIniciando la extracción detallada de los restaurantes de la página de Tripadvisor...")
restaurantes_detallados = []

restaurant_blocks = soup_tripadvisor.find_all('div', {'data-automation': 'restaurantCard'})

if not restaurant_blocks:
    print("AVISO: No se encontró ningún bloque principal de restaurantes con los selectores configurados (('data-automation': 'restaurantCard')).")
    print("Por favor, **REVISE LA INSPECCIÓN MANUAL** si el diseño de la página cambió y actualice este selector.")

top_n_restaurants = restaurant_blocks[:5]

for block in top_n_restaurants:
    nombre_link_tag = block.find('a', class_=lambda c: c and 'BMQDV' in c and 'ukgos' in c)
    nombre = "Nombre no encontrado"
    if nombre_link_tag:
        nombre_div_tag = nombre_link_tag.find('div', class_=lambda c: c and 'biGQs' in c and 'OgHOe' in c)
        if nombre_div_tag:
            nombre = nombre_div_tag.get_text(strip=True)
            
    reviews_tag = block.find('div', {'data-automation': 'bubbleReviewCount'})
    reviews = reviews_tag.find('span').get_text(strip=True) if reviews_tag and reviews_tag.find('span') else "Reseñas no encontradas"

    rating_tag = block.find('div', {'data-automation': 'bubbleRatingValue'})
    rating = rating_tag.find('span').get_text(strip=True) if rating_tag and rating_tag.find('span') else "Calificación no encontrada"

    culinaria_precio_div = block.find('div', class_=lambda c: c and 'Zvrsw' in c and 'biqBm' in c)

    tipo_culinaria = "Tipo de cocina no encontrado"
    precio = "Precio no encontrado"

    if culinaria_precio_div:
        spans_info = culinaria_precio_div.find_all('span',
            class_=lambda c: c and 'biGQs' in c and 'ZNjNf' in c
        )
        
        if len(spans_info) >= 1:
            tipo_culinaria = spans_info[0].get_text(strip=True)
            
        for i in range(1, len(spans_info)):
            text = spans_info[i].get_text(strip=True)
            if '$' in text:
                precio = text
                break
    
    localizacion = "Ubicación no especificada (desde el nombre)"
    if ' - ' in nombre:
        partes_nombre = nombre.split(' - ')
        if len(partes_nombre) > 1:
            localizacion = partes_nombre[-1].strip()
            
    link = "Enlace no encontrado"
    if nombre_link_tag and 'href' in nombre_link_tag.attrs:
        link = "https://www.tripadvisor.com.mx" + nombre_link_tag['href']

    restaurantes_detallados.append({
        "Nombre": nombre,
        "Calificación": rating,
        "Reseñas": reviews,
        "Precio": precio,
        "Tipo de cocina": tipo_culinaria,
        "Ubicacion": localizacion,
        "Enlace": link
    })

if restaurantes_detallados:
    print(f"\n--- {len(restaurantes_detallados)} Restaurantes Extraidos de Tripadvisor ---")
    for i, r in enumerate(restaurantes_detallados):
        print(f"Restaurante #{i+1}:")
        print(f"  Nombre: {r['Nombre']}")
        print(f"  Calificación: {r['Calificación']}")
        print(f"  Reseñas: {r['Reseñas']}")
        print(f"  Precio: {r['Precio']}")
        print(f"  Tipo de Cocina: {r['Tipo de cocina']}")
        print(f"  Ubicacion: {r['Ubicacion']}")
        print(f"  Enlace: {r['Enlace']}")
        print("*" * 40)
else:
    print("No se extrajo ningún detalle de restaurantes. **Verifique los selectores HTML en el bloque de raspado**.")

print("*" * 50)